In [7]:
#working on the train_set
#importing the libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
print("Libraries imported successfully")


Libraries imported successfully


In [8]:

#now we are converting our images to csv format that are already in jpg
import os
from PIL import Image

# Path to your "train_set" folder
image_dir = '/Users/apple/Desktop/Cats VS Dogs/training_set'
data = []


label_map = {'cats': 0, 'dogs': 1}  # Map folder names to labels

for label_name in os.listdir(image_dir):
    label_path = os.path.join(image_dir, label_name)
    
    if os.path.isdir(label_path) and label_name in label_map:
        label = label_map[label_name]
        
        for file in os.listdir(label_path):
            if file.endswith('.jpg'):
                img_path = os.path.join(label_path, file)
                
                try:
                    image = Image.open(img_path).convert('L')  # grayscale
                    image = image.resize((28, 28))             # resize to 28x28
                    pixels = np.array(image).flatten()         # flatten to 784
                    row = [label] + pixels.tolist()
                    data.append(row)
                except:
                    print(f"Error reading {img_path}")

# Create column names: label, pixel0, pixel1, ..., pixel783
columns = ['label'] + [f'pixel{i}' for i in range(784)]

# Convert to DataFrame and save
df = pd.DataFrame(data, columns= columns )
df.to_csv('cats_vs_dogs(train).csv', index=False)


In [9]:
torch.manual_seed(42)  # For reproducibility
raw_df = pd.read_csv('cats_vs_dogs(train).csv')
X_train= raw_df.drop('label', axis=1).values
y_train = raw_df['label'].values


In [10]:




#now we will create a custom dataset class
class CatsVsDogsDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = torch.tensor(X, dtype =torch.float32)
        self.y= torch.tensor(y,dtype=torch.long)

    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]
    
# Create the dataset
train_dataset = CatsVsDogsDataset(X_train, y_train)
# Create the DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)



In [11]:
#now we will create our neural network model
learning_rate = 0.01
n_epochs = 25
class SimpleNN(nn.Module):
    def __init__(self, n_features=784, n_classes=2):
        super().__init__()
        self.model =  nn.Sequential(
            nn.Linear(n_features, 128),  # Input layer to hidden layer
            nn.ReLU(),                   # Activation function
            nn.Linear(128, 64),    # Hidden layer 2
            nn.ReLU(),                   # Activation function  
            nn.Linear(64, n_classes)     # Output layer
        )
        
    def forward(self, x):
        
        
        return self.model(x)
        
        
  
  

In [12]:
  # Initialize the model, loss function, and optimizer
model = SimpleNN(X_train.shape[1])
criterion = nn.CrossEntropyLoss()  # For multi-class classification
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

# Training loop
for epoch in range (n_epochs):
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs,batch_y)
        optimizer.zero_grad()  # Zero the gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update the weights
        total_loss += loss.item()  
    avg_loss = total_loss/len(train_loader)
    print(f'Epoch: {epoch + 1}, Loss: {avg_loss}')

    # Save the trained model
    torch.save(model.state_dict(), 'cats_vs_dogs_model.pth')




Epoch: 1, Loss: 1.6399274158477783
Epoch: 2, Loss: 0.7377981100082397
Epoch: 3, Loss: 0.6867978699207306
Epoch: 4, Loss: 0.6943960733413697
Epoch: 5, Loss: 0.6756534810066224
Epoch: 6, Loss: 0.6777565493583679
Epoch: 7, Loss: 0.6933142439126968
Epoch: 8, Loss: 0.6763015351295472
Epoch: 9, Loss: 0.675350759267807
Epoch: 10, Loss: 0.6719732451438903
Epoch: 11, Loss: 0.6740423748493195
Epoch: 12, Loss: 0.6834478123188019
Epoch: 13, Loss: 0.6842364008426667
Epoch: 14, Loss: 0.6723714592456818
Epoch: 15, Loss: 0.6731180443763732
Epoch: 16, Loss: 0.6732662253379822
Epoch: 17, Loss: 0.6800588240623474
Epoch: 18, Loss: 0.6686340453624725
Epoch: 19, Loss: 0.665741409778595
Epoch: 20, Loss: 0.6677397809028626
Epoch: 21, Loss: 0.662906625509262
Epoch: 22, Loss: 0.6655531685352325
Epoch: 23, Loss: 0.6639089090824127
Epoch: 24, Loss: 0.662014963388443
Epoch: 25, Loss: 0.6561807918548584
